# LIH

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# Reproducible fluctuations
rng = np.random.default_rng(42)

file_path = "LIH2_MV_Costumer.xlsx"

# Read second sheet
df = pd.read_excel(file_path, sheet_name=1)

# Rename Swedish headers to easier internal names
df = df.rename(columns={
    "Datum": "start_time",
    df.columns[1]: "end_time",
    "Aktiv effekt (kW)": "p_kw",
    "Reaktiv effekt (kvar)": "q_kvar",
    "Skenbar effekt (kVA)": "s_kva",
})

# Parse Excel datetimes
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])

rows_15min = []

for _, row in df.iterrows():
    hour_start = row["start_time"]

    # Four 15-minute timestamps inside the hour
    quarter_times = pd.date_range(hour_start, periods=4, freq="15min")

    # Create small fluctuations around 1.0 and normalize so the mean stays exactly 1.0
    p_factors = rng.uniform(0.92, 1.08, 4)
    p_factors = p_factors / p_factors.mean()

    q_factors = rng.uniform(0.92, 1.08, 4)
    q_factors = q_factors / q_factors.mean()

    # Convert kW/kvar to MW/MVAr before storing the 15-minute profile
    p_values = (row["p_kw"] * p_factors) / 1000.0
    q_values = (row["q_kvar"] * q_factors) / 1000.0
    s_values = np.sqrt(p_values**2 + q_values**2)

    for t, p, q, s in zip(quarter_times, p_values, q_values, s_values):
        rows_15min.append({
            "timestamp": t,
            "Aktiv effekt (MW)": p,
            "Reaktiv effekt (MVAr)": q,
            "Skenbar effekt (MVA)": s,
        })

df_15min = pd.DataFrame(rows_15min)

# Optional: round values for readability
df_15min["Aktiv effekt (MW)"] = df_15min["Aktiv effekt (MW)"].round(6)
df_15min["Reaktiv effekt (MVAr)"] = df_15min["Reaktiv effekt (MVAr)"].round(6)
df_15min["Skenbar effekt (MVA)"] = df_15min["Skenbar effekt (MVA)"].round(6)

display(df_15min.head(12))

,timestamp,Aktiv effekt (MW),Reaktiv effekt (MVAr),Skenbar effekt (MVA)
0,2025-03-08 00:00:00,0.093168,0.0,0.093168
1,2025-03-08 00:15:00,0.088382,0.0,0.088382
2,2025-03-08 00:30:00,0.094376,0.0,0.094376
3,2025-03-08 00:45:00,0.092074,0.0,0.092074
4,2025-03-08 01:00:00,0.087902,0.0,0.087902
5,2025-03-08 01:15:00,0.092721,0.0,0.092721
6,2025-03-08 01:30:00,0.091531,0.0,0.091531
7,2025-03-08 01:45:00,0.099845,0.0,0.099845
8,2025-03-08 02:00:00,0.094527,0.0,0.094527
9,2025-03-08 02:15:00,0.087169,0.0,0.087169


In [8]:
import pandas as pd

# Ensure timestamp is datetime
df_15min["timestamp"] = pd.to_datetime(df_15min["timestamp"])

# Split dataset
part_2026 = df_15min[df_15min["timestamp"].dt.year == 2026].copy()
part_2025 = df_15min[df_15min["timestamp"].dt.year == 2025].copy()

# Move 2026 chunk back by one year so it becomes Jan-Mar 2025
part_2026["timestamp"] = part_2026["timestamp"] - pd.DateOffset(years=1)

# Concatenate in calendar order
df_2025_full = pd.concat([part_2026, part_2025], ignore_index=True)
df_2025_full = df_2025_full.sort_values("timestamp").reset_index(drop=True)

print(df_2025_full["timestamp"].min())
print(df_2025_full["timestamp"].max())
print(len(df_2025_full))

display(df_2025_full.head(8))
display(df_2025_full.tail(8))

2025-01-01 00:00:00
2025-12-31 23:45:00
35040


,timestamp,Aktiv effekt (MW),Reaktiv effekt (MVAr),Skenbar effekt (MVA)
0,2025-01-01 00:00:00,0.075152,0.0,0.075152
1,2025-01-01 00:15:00,0.080223,0.0,0.080223
2,2025-01-01 00:30:00,0.079579,0.0,0.079579
3,2025-01-01 00:45:00,0.077045,0.0,0.077045
4,2025-01-01 01:00:00,0.075092,0.0,0.075092
5,2025-01-01 01:15:00,0.078080,0.0,0.078080
6,2025-01-01 01:30:00,0.072222,0.0,0.072222
7,2025-01-01 01:45:00,0.082606,0.0,0.082606


,timestamp,Aktiv effekt (MW),Reaktiv effekt (MVAr),Skenbar effekt (MVA)
35032,2025-12-31 22:00:00,0.074459,0.0,0.074459
35033,2025-12-31 22:15:00,0.075781,0.0,0.075781
35034,2025-12-31 22:30:00,0.075466,0.0,0.075466
35035,2025-12-31 22:45:00,0.086294,0.0,0.086294
35036,2025-12-31 23:00:00,0.076052,0.0,0.076052
35037,2025-12-31 23:15:00,0.079099,0.0,0.079099
35038,2025-12-31 23:30:00,0.076414,0.0,0.076414
35039,2025-12-31 23:45:00,0.076435,0.0,0.076435


In [9]:
output_path = "LIH2_MV_Costumer_15min.xlsx"
df_2025_full.to_excel(output_path, index=False)
print(f"Saved to {output_path}")

Saved to LIH2_MV_Costumer_15min.xlsx


# Cargo Network

Read `inputs/cargo_grid_loads.xlsx` (hourly) and build a 15-minute profile for each sheet.  
For every hour, the four quarter-hour values fluctuate randomly but their **mean equals the original hourly value** (active and reactive power). Apparent power is recomputed from the quarter-hour P and Q.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Paths relative to this notebook folder (network_data/)
ROOT = Path(".").resolve()
INPUT_XLSX = ROOT / "inputs" / "cargo_grid_loads.xlsx"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_XLSX = OUTPUT_DIR / "15min_cargo_grid_loads.xlsx"

P_COL = "Aktiv effekt (kW)"
Q_COL = "Reaktiv effekt (kvar)"
S_COL = "Skenbar effekt (kVA)"
P_MW_COL = "Aktiv effekt (MW)"
Q_MW_COL = "Reaktiv effekt (MVAr)"
S_MW_COL = "Skenbar effekt (MVA)"
KW_TO_MW = 1000.0

RNG_SEED = 42
SPREAD = 0.08  # factors in [1-SPREAD, 1+SPREAD], then mean-normalized to 1


def _mean_preserving_factors(rng: np.random.Generator, n: int) -> np.ndarray:
    factors = rng.uniform(1.0 - SPREAD, 1.0 + SPREAD, n)
    return factors / factors.mean()


CALENDAR_15MIN_LEN = 35040  # 8760 h × 4 quarters
CALENDAR_15MIN_INDEX = pd.date_range("2025-01-01 00:00:00", periods=CALENDAR_15MIN_LEN, freq="15min")


def align_to_calendar_grid(df_15: pd.DataFrame) -> pd.DataFrame:
    """One row per 15-min slot; fill rare DST gaps by time interpolation."""
    out = df_15.set_index("Datum").sort_index()
    out = out[~out.index.duplicated(keep="last")]
    out = out.reindex(CALENDAR_15MIN_INDEX)
    out[[P_COL, Q_COL, S_COL]] = out[[P_COL, Q_COL, S_COL]].interpolate(method="time")
    out = out.reset_index().rename(columns={"index": "Datum"})
    out["Datum.1"] = out["Datum"] + pd.Timedelta(minutes=15)
    return out


def hourly_sheet_to_15min(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    """Expand one hourly sheet to 15-minute resolution (fixed 4 steps per hour)."""
    work = df.copy()
    work["Datum"] = pd.to_datetime(work["Datum"])
    if "Datum.1" in work.columns:
        work["Datum.1"] = pd.to_datetime(work["Datum.1"])
    work = work.sort_values("Datum", kind="mergesort").reset_index(drop=True)

    rows_15 = []
    for _, row in work.iterrows():
        period_start = row["Datum"]

        # Fixed calendar grid: exactly four 15-minute slots per hourly row
        quarter_times = pd.date_range(period_start, periods=4, freq="15min")
        n_steps = 4
        p_factors = _mean_preserving_factors(rng, n_steps)
        q_factors = _mean_preserving_factors(rng, n_steps)

        p_values = row[P_COL] * p_factors
        q_values = row[Q_COL] * q_factors
        s_values = np.sqrt(p_values**2 + q_values**2)

        for t, p, q, s in zip(quarter_times, p_values, q_values, s_values):
            rows_15.append(
                {
                    "Datum": t,
                    "Datum.1": t + pd.Timedelta(minutes=15),
                    P_COL: p,
                    Q_COL: q,
                    S_COL: s,
                }
            )

    return align_to_calendar_grid(pd.DataFrame(rows_15))


def add_mw_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add MW / MVAr / MVA columns next to the kW / kvar / kVA columns."""
    out = df.copy()
    out[P_MW_COL] = out[P_COL] / KW_TO_MW
    out[Q_MW_COL] = out[Q_COL] / KW_TO_MW
    out[S_MW_COL] = out[S_COL] / KW_TO_MW
    col_order = [
        "Datum",
        "Datum.1",
        P_COL,
        P_MW_COL,
        Q_COL,
        Q_MW_COL,
        S_COL,
        S_MW_COL,
    ]
    return out[[c for c in col_order if c in out.columns]]


xl = pd.ExcelFile(INPUT_XLSX)
rng_master = np.random.default_rng(RNG_SEED)
sheets_15min = {}

for sheet_name in xl.sheet_names:
    df_hourly = pd.read_excel(INPUT_XLSX, sheet_name=sheet_name)

    # Metadata sheet (no hourly time series)
    if "Datum" not in df_hourly.columns or df_hourly["Datum"].dtype == object:
        sheets_15min[sheet_name] = df_hourly
        continue

    sheet_rng = np.random.default_rng(rng_master.integers(0, 2**32 - 1))
    df_15 = add_mw_columns(hourly_sheet_to_15min(df_hourly, sheet_rng))
    sheets_15min[sheet_name] = df_15

    # Quick check: each original row's 15-min mean must match its hourly P
    hourly_p = df_hourly[P_COL].to_numpy(dtype=float)
    means = [
        df_15[P_COL].iloc[i * 4 : (i + 1) * 4].mean()
        for i in range(len(df_hourly))
    ]
    check = np.max(np.abs(np.array(means) - hourly_p))
    print(
        f"{sheet_name}: {len(df_hourly)} h -> {len(df_15)} x 15min | "
        f"max |mean(P_15min)-P_hourly| = {check:.2e}"
    )
    if len(df_15) != CALENDAR_15MIN_LEN or df_15["Datum"].nunique() != CALENDAR_15MIN_LEN:
        raise ValueError(f"{sheet_name}: expected {CALENDAR_15MIN_LEN} unique 15-min rows")
    if df_15["Datum"].iloc[0] != CALENDAR_15MIN_INDEX[0] or df_15["Datum"].iloc[-1] != CALENDAR_15MIN_INDEX[-1]:
        raise ValueError(f"{sheet_name}: timestamp range does not match calendar grid")

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    for sheet_name, df_out in sheets_15min.items():
        df_out.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\nSaved: {OUTPUT_XLSX}")
sheets_15min["OGP"].head(8)

OGP: 8760 h -> 35040 x 15min | max |mean(P_15min)-P_hourly| = 2.78e-17 | unique timestamps = 35036
TBO: 8760 h -> 35040 x 15min | max |mean(P_15min)-P_hourly| = 1.42e-14 | unique timestamps = 35036
URA: 8760 h -> 35040 x 15min | max |mean(P_15min)-P_hourly| = 1.42e-14 | unique timestamps = 35036
SVN: 8760 h -> 35040 x 15min | max |mean(P_15min)-P_hourly| = 5.68e-14 | unique timestamps = 35036
LAX T1: 8760 h -> 35040 x 15min | max |mean(P_15min)-P_hourly| = 2.84e-14 | unique timestamps = 35036
LAX T2: 8760 h -> 35040 x 15min | max |mean(P_15min)-P_hourly| = 2.84e-14 | unique timestamps = 35036
KBV: 8760 h -> 35040 x 15min | max |mean(P_15min)-P_hourly| = 5.68e-14 | unique timestamps = 35040
NSG: 8760 h -> 35040 x 15min | max |mean(P_15min)-P_hourly| = 5.68e-14 | unique timestamps = 35036
KLU: 8760 h -> 35040 x 15min | max |mean(P_15min)-P_hourly| = 2.84e-14 | unique timestamps = 35036

Saved: /Users/michelangeloinze/Desktop/Master_Thesis/coding_MichelangeloInzerilli_MScThesis/network_da

,Datum,Datum.1,Aktiv effekt (kW),Aktiv effekt (MW),Reaktiv effekt (kvar),Reaktiv effekt (MVAr),Skenbar effekt (kVA),Skenbar effekt (MVA)
0,2025-01-01 00:00:00,2025-01-01 00:15:00,0.107838,0.000108,0.068078,0.000068,0.127528,0.000128
1,2025-01-01 00:15:00,2025-01-01 00:30:00,0.108020,0.000108,0.063259,0.000063,0.125180,0.000125
2,2025-01-01 00:30:00,2025-01-01 00:45:00,0.111003,0.000111,0.061350,0.000061,0.126828,0.000127
3,2025-01-01 00:45:00,2025-01-01 01:00:00,0.109140,0.000109,0.067314,0.000067,0.128229,0.000128
4,2025-01-01 01:00:00,2025-01-01 01:15:00,0.116474,0.000116,0.062482,0.000062,0.132175,0.000132
5,2025-01-01 01:15:00,2025-01-01 01:30:00,0.111028,0.000111,0.065933,0.000066,0.129129,0.000129
6,2025-01-01 01:30:00,2025-01-01 01:45:00,0.108143,0.000108,0.069565,0.000070,0.128585,0.000129
7,2025-01-01 01:45:00,2025-01-01 02:00:00,0.116356,0.000116,0.066020,0.000066,0.133781,0.000134
